# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [458]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [459]:
filepath = "data/AviationData.csv"
dataset = pd.read_csv(filepath,encoding="latin-1", parse_dates=["Event.Date","Publication.Date"])

dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                88889 non-null  object        
 1   Investigation.Type      88889 non-null  object        
 2   Accident.Number         88889 non-null  object        
 3   Event.Date              88889 non-null  datetime64[ns]
 4   Location                88837 non-null  object        
 5   Country                 88663 non-null  object        
 6   Latitude                34382 non-null  object        
 7   Longitude               34373 non-null  object        
 8   Airport.Code            50132 non-null  object        
 9   Airport.Name            52704 non-null  object        
 10  Injury.Severity         87889 non-null  object        
 11  Aircraft.damage         85695 non-null  object        
 12  Aircraft.Category       32287 non-null  object

/var/folders/j7/sf1mql6927x06g0x4zv_tm7r0000gn/T/ipykernel_73329/2380076579.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv(filepath,encoding="latin-1", parse_dates=["Event.Date","Publication.Date"])
/var/folders/j7/sf1mql6927x06g0x4zv_tm7r0000gn/T/ipykernel_73329/2380076579.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dataset = pd.read_csv(filepath,encoding="latin-1", parse_dates=["Event.Date","Publication.Date"])


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [460]:
#Relevant Columns for analysis
relevant = ["Event.Date","Injury.Severity", "Aircraft.damage", "Make","Model", "Number.of.Engines", "Engine.Type","Purpose.of.flight","Total.Fatal.Injuries","Total.Serious.Injuries","Total.Minor.Injuries", "Total.Uninjured", "Weather.Condition", "Broad.phase.of.flight"]

#Filter by Publication date and whether the craft is professionally built
#Since the client is a airline/airplane insurer it can be assumed that they would not care about Balloons, Gliders, or Helicopters in the data.
filtered_data = dataset.loc[(dataset["Publication.Date"].dt.year > 1983) & (dataset["Amateur.Built"] == "No") & (dataset["Aircraft.Category"] == "Airplane"),relevant]

#Check for N/A values in filtered data
filtered_data.isna().sum()


Event.Date                    0
Injury.Severity             617
Aircraft.damage            1062
Make                          1
Model                        16
Number.of.Engines          1970
Engine.Type                3135
Purpose.of.flight          3139
Total.Fatal.Injuries       2644
Total.Serious.Injuries     2656
Total.Minor.Injuries       2371
Total.Uninjured             583
Weather.Condition          2364
Broad.phase.of.flight     17662
dtype: int64

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [461]:


#Get the injury metrics and factor the N/A as 0.0
#I made the assumtion that the N/A were 0.0s because they correspond to Non-fatal and Incident serverity accidents where a value is present for at least on other type
filtered_data["Total.Fatal.Serious"] = filtered_data["Total.Fatal.Injuries"].fillna(0.0) + filtered_data["Total.Serious.Injuries"].fillna(0.0)
filtered_data["Total.People"] = filtered_data["Total.Fatal.Serious"] + filtered_data["Total.Minor.Injuries"].fillna(0.0) + filtered_data["Total.Uninjured"].fillna(0.0)
filtered_data["Critical.Injury.Rate"] = filtered_data["Total.Fatal.Serious"] / filtered_data["Total.People"]



**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [462]:
#The Aircraft.damage column has n/a values and those need to be changed.
# Since there is a unknown value in this column I changed all N/A's to Unknown since I dont have enough context from the data to assume anything else.
filtered_data["Plane.Destroyed"] = filtered_data["Aircraft.damage"].fillna("Unknown") == "Destroyed"


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

### Format Column to Title-Case and Strip Whitespace


In [463]:
#Here I'm getting rid of whitespace, formating to title-case, and replacing the periods after some company names.
filtered_data["Make"] = filtered_data["Make"].str.title().str.strip().str.replace(".", "", regex=False)

### Make Titles Cohesive

In [464]:
#These were found by checking the first 60 values from head() and the last 60 from tail()
make_mapping = {
    "Learjet Inc": "Learjet",
    "Gates Lear Jet Corp.": "Learjet",
    "Gates Learjet": "Learjet",

    "Mooney Aircraft Corp": "Mooney",

    "Rockwell International": "Rockwell",

    "Cirrus Design Corp": "Cirrus",

    "Mcdonnell Douglas": "Douglas",

    "Sabreliner Corp": "Sabreliner",

    "Aviat Aircraft Inc": "Aviat",

    "Airbus Industrie": "Airbus",

    "Rotorsport Uk Ltd": "RotorSport",

    "Bombardier Inc": "Bombardier",
    "Bombardier Learjet Corp": "Bombardier",

    "Air Tractor Inc": "Air Tractor",
    "Air Tractor, Inc": "Air Tractor",
    "Diamond Aircraft Ind Inc": "Diamond",

    "Dehavilland": "De Havilland",

    "Piper Aircraft Inc": "Piper",
    "Piper Aircraft Co": "Piper",

    "Cessna Aircraft Co": "Cessna",

    "Grumman American": "Grumman",
    "Grumman American Avn Corp": "Grumman",
    "Grumman Acft Eng Cor-Schweizer": "Grumman",

    "Ayres Corporation": "Ayres",

    "Flight Design Gmbh": "Flight Design",

}

filtered_data["Make"] = filtered_data["Make"].replace(make_mapping)




### Filter Makes with under 50 sightings

In [ ]:
#Get the count per make
make_counts = filtered_data["Make"].value_counts()

#Filter makes with less then 50 sightings in the data
filtered_data = filtered_data[
    filtered_data["Make"].isin(make_counts[make_counts >= 50].index)
]


Make
Cessna                        6908
Piper                         3891
Beech                         1387
Boeing                        1088
Air Tractor                    419
Mooney                         384
Cirrus                         374
Grumman                        276
Airbus                         242
Bellanca                       216
Maule                          210
Aeronca                        195
De Havilland                   163
Champion                       156
Aviat                          145
Embraer                        141
Luscombe                       139
Stinson                        129
Douglas                        119
Diamond                        119
Bombardier                     108
North American                 102
Rockwell                        98
Ayres                           95
Taylorcraft                     93
Aero Commander                  83
Flight Design                   75
Socata                          70
Raytheon Aircra

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
filtered_data = filtered_data.dropna(subset=["Model"])

filtered_data.tail(50)
#Checks if Models are unique to Make.
model_make_counts = filtered_data.groupby("Model")["Make"].nunique()

model_make_counts.sort_values(ascending=False).head()

#They are not and thus we make a unique identifier
#I thought of this as kind of like make a cache key
filtered_data["Aircraft.Type"] = (filtered_data["Make"] + " " + filtered_data["Model"])



,Event.Date,Injury.Severity,Aircraft.damage,Make,Model,Number.of.Engines,Engine.Type,Purpose.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Total.Fatal.Serious,Total.People,Critical.Injury.Rate,Plane.Destroyed,Aircraft.Type
5,1979-09-17,Non-Fatal,Substantial,Douglas,DC9,2.0,Turbo Fan,NaN,NaN,NaN,1.0,44.0,VMC,Climb,0.0,45.0,0.00,False,Douglas DC9
260,1982-02-07,Non-Fatal,Substantial,Bellanca,7KCAB,1.0,Reciprocating,Personal,NaN,NaN,NaN,2.0,VMC,Landing,0.0,2.0,0.00,False,Bellanca 7KCAB
351,1982-02-19,Incident,NaN,Boeing,B-727-200,3.0,Turbo Jet,NaN,NaN,NaN,NaN,83.0,IMC,Landing,0.0,83.0,0.00,False,Boeing B-727-200
593,1982-03-16,Fatal(1),Substantial,Beech,C24R,1.0,Reciprocating,Unknown,1.0,NaN,NaN,NaN,IMC,Landing,1.0,1.0,1.00,False,Beech C24R
801,1982-04-09,Non-Fatal,Substantial,Piper,PA-23-170,2.0,Reciprocating,Instructional,NaN,NaN,NaN,2.0,VMC,Standing,0.0,2.0,0.00,False,Piper PA-23-170
1164,1982-05-12,Incident,NaN,Learjet,24B,2.0,Turbo Jet,Business,NaN,NaN,NaN,3.0,VMC,Climb,0.0,3.0,0.00,False,Learjet 24B
1298,1982-05-22,Non-Fatal,Substantial,Bellanca,7GCBC,1.0,Reciprocating,Personal,NaN,NaN,NaN,1.0,VMC,Climb,0.0,1.0,0.00,False,Bellanca 7GCBC
1329,1982-05-25,Non-Fatal,Substantial,Luscombe,8E,1.0,Reciprocating,Personal,NaN,NaN,NaN,1.0,VMC,Landing,0.0,1.0,0.00,False,Luscombe 8E
1834,1982-07-05,Incident,Minor,Boeing,727-233,3.0,Turbo Fan,NaN,NaN,NaN,NaN,74.0,VMC,Climb,0.0,74.0,0.00,False,Boeing 727-233
1970,1982-07-16,Non-Fatal,Destroyed,Cessna,172P,1.0,Reciprocating,Personal,NaN,1.0,3.0,NaN,VMC,Landing,1.0,4.0,0.25,True,Cessna 172P


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [467]:
#Just doing some basic cleanup for the other columns, removing whitespace and using title-case

filtered_data["Engine.Type"] = (
    filtered_data["Engine.Type"]
    .str.title()
    .str.strip()
)
#Readibility Fix 
filtered_data["Weather.Condition"] = filtered_data["Weather.Condition"].replace({
    "UNK": "Unknown"
})

filtered_data["Purpose.of.flight"] = (
    filtered_data["Purpose.of.flight"]
    .str.title()
    .str.strip()
)

filtered_data["Broad.phase.of.flight"] = (
    filtered_data["Broad.phase.of.flight"]
    .str.title()
    .str.strip()
)
#Incidents without people aren't really useful for analysis
filtered_data = filtered_data[filtered_data["Total.People"] > 1]

# It shouldn't be possible for a airplane to have 0 engines
filtered_data = filtered_data[filtered_data["Number.of.Engines"] > 0]

filtered_data[filtered_data['Number.of.Engines'] == 0]

,Event.Date,Injury.Severity,Aircraft.damage,Make,Model,Number.of.Engines,Engine.Type,Purpose.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Total.Fatal.Serious,Total.People,Critical.Injury.Rate,Plane.Destroyed,Aircraft.Type


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [468]:
#The broad phase of flight was 84% Nan so it gets dropped.
filtered_data = filtered_data.drop(columns=["Broad.phase.of.flight"])
na_percent = filtered_data.isna().mean()

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [469]:
filtered_data.to_csv("cleaned_aviation_data.csv", index=False)